# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bilalahmed251/-ML-Search-Discovery/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# I will inspect the distributions of the key baseline signals before judging them. I will compare medians and percentiles because some fields may have heavy right tails, where a small number of pages have unusually large values. This helps avoid making decisions based only on misleading averages.


In [16]:
import os
import pandas as pd
import numpy as np

repo = "/content/ML-Search-Discovery"

if not os.path.exists(repo):
    !git clone -q --depth 1 https://github.com/bilalahmed251/-ML-Search-Discovery.git {repo}

df = pd.read_csv(
    f"{repo}/data/raw/content_refresh_anonymized.csv"
 ).copy()

key_fields = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "word_count",
]

available_fields = [
    field for field in key_fields
    if field in df.columns
]

summary = df[available_fields].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.99]
).T

summary["missing"] = df[available_fields].isna().sum()
summary["skewness"] = df[available_fields].skew()

print("Rows:", len(df))
print("Fields checked:", available_fields)
display(summary)

print("Median values:")
display(df[available_fields].median())

print("99th-percentile values:")
display(df[available_fields].quantile(0.99))


Rows: 30000
Fields checked: ['impressions_90d', 'ctr', 'avg_position', 'content_age_days', 'days_since_last_update', 'word_count']


,count,mean,std,min,25%,50%,75%,90%,99%,max,missing,skewness
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.00,3615.25,12136.40,73505.830,517715.0,0,11.384919
ctr,30000.0,0.510733,3.279162,0.0,0.0,0.07,0.29,0.65,8.330,100.0,0,17.444252
avg_position,30000.0,16.342380,15.216790,0.0,6.2,10.80,22.30,36.80,69.901,245.0,0,1.984214
content_age_days,30000.0,256.167800,132.707930,90.0,132.0,236.00,333.00,463.00,537.000,564.0,0,0.489000
days_since_last_update,30000.0,46.098300,42.078709,1.0,20.0,20.00,104.00,104.00,106.000,373.0,0,1.161283
word_count,22301.0,3107.760325,1452.382598,8.0,2413.0,2877.00,3666.00,5327.00,7292.000,9546.0,7699,0.937935


Median values:


,0
impressions_90d,731.00
ctr,0.07
avg_position,10.80
content_age_days,236.00
days_since_last_update,20.00
word_count,2877.00


99th-percentile values:


,0.99
impressions_90d,73505.830
ctr,8.330
avg_position,69.901
content_age_days,537.000
days_since_last_update,106.000
word_count,7292.000


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Impressions and CTR have strong positive skewness and large gaps between their medians and 99th percentiles, showing heavy right tails. Average position is also right-skewed, while content age and days since last update are less skewed. Word count has 7,699 missing values, so it should be handled carefully. Percentile ranks are more appropriate than raw values for the baseline score.


In [18]:
df["is_declining"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

df["visibility_group"] = np.where(
    df["impressions_90d"] >= df["impressions_90d"].median(),
    "high_visibility",
    "low_visibility"
)

df["position_group"] = np.where(
    df["avg_position"] >= df["avg_position"].median(),
    "weaker_position",
    "stronger_position"
)

df["age_group"] = np.where(
    df["content_age_days"] >= df["content_age_days"].median(),
    "older_content",
    "newer_content"
)

print("Visibility test:")
display(df.groupby("visibility_group")["is_declining"].agg(
    declining_rate="mean",
    pages="count"
))

print("Position test:")
display(df.groupby("position_group")["is_declining"].agg(
    declining_rate="mean",
    pages="count"
))

print("Age test:")
display(df.groupby("age_group")["is_declining"].agg(
    declining_rate="mean",
    pages="count"
))


Visibility test:


,declining_rate,pages
visibility_group,,
high_visibility,0.593989,15007
low_visibility,0.490095,14993


Position test:


,declining_rate,pages
position_group,,
stronger_position,0.520358,14982
weaker_position,0.563724,15018


Age test:


,declining_rate,pages
age_group,,
newer_content,0.62522,14790
older_content,0.46121,15210


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Signal 1 — Visibility: CONFIRMED. High-visibility pages have a higher declining rate than low-visibility pages (0.594 vs 0.490).

# Signal 2 — Average position: CONFIRMED, but weak. Weaker-position pages have a slightly higher declining rate than stronger-position pages (0.564 vs 0.520), so the relationship is directional but modest.

# Signal 3 — Content age: OPPOSITE. Newer-content pages have a higher declining rate than older-content pages (0.625 vs 0.461). Therefore, the assumption that older pages are more likely to decline is not supported in this dataset.



## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# The audit suggests that visibility is a useful directional signal because high-visibility pages show a higher declining rate. However, average position is only a weak signal, and the age signal shows the opposite pattern from our original assumption. The content team should use the baseline as a review shortlist, not as an automatic refresh decision.



In [21]:
# Recreate the HIGH_VISIBILITY flag used by the baseline
visibility_cutoff = df["impressions_90d"].quantile(0.75)

df["HIGH_VISIBILITY"] = (
    df["impressions_90d"] >= visibility_cutoff
)

flag_test = df.groupby("HIGH_VISIBILITY")["is_declining"].agg(
    declining_rate="mean",
    pages="count"
)

print("HIGH_VISIBILITY cutoff:", visibility_cutoff)
display(flag_test)

high_rate = flag_test.loc[True, "declining_rate"]
other_rate = flag_test.loc[False, "declining_rate"]

if high_rate > other_rate:
    print("Verdict: CONFIRMED")
else:
    print("Verdict: OPPOSITE")

HIGH_VISIBILITY cutoff: 3615.25


,declining_rate,pages
HIGH_VISIBILITY,,
False,0.535422,22500
True,0.562000,7500


Verdict: CONFIRMED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.